# Managed vs External Tables — And Why Your File Got Deleted
**Day 4 | The most important concept to get right before writing any data**

---

## Why Did the File Get Deleted from ADLS When You Deleted It from the Volume UI?

Short answer: **because a Volume IS your ADLS path — they are the same storage.**

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
                    │
                    │  Unity Catalog maps this path to:
                    ▼
abfss://bronze@evdatalakedev.dfs.core.windows.net/
```

The Volume is not a copy of your ADLS data — it is a **pointer** to it.
When you delete a file through the Volume UI, you are deleting the actual file
on ADLS. There is no recycle bin. The file is gone.

This is because `bronze-volume` is an **External Volume** — Unity Catalog registered
your ADLS path as a Volume, but it does NOT own the data or protect it from deletion.

---

## The Full Picture: Managed vs External — Volumes AND Tables

This concept applies to both **Volumes** and **Tables**. In both cases, the question is:
**who owns the underlying data files?**

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    MANAGED (Unity Catalog owns the data)                    │
│                                                                             │
│   You: CREATE TABLE pipeline_audit (...)                                    │
│   UC:  stores files at  →  abfss://uc-root/.../pipeline_audit/              │
│                                                                             │
│   DROP TABLE pipeline_audit  →  metadata deleted + FILES DELETED FROM ADLS │
│   DROP SCHEMA default        →  all tables + all files deleted              │
│                                                                             │
│   Use when: data lives only inside Databricks, no external tool reads files │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│                    EXTERNAL (YOU own the data)                              │
│                                                                             │
│   You: CREATE TABLE charging_sessions LOCATION 'abfss://silver@.../cs/'    │
│   UC:  only stores metadata → the actual files stay at YOUR ADLS path      │
│                                                                             │
│   DROP TABLE charging_sessions  →  metadata deleted, FILES STAY ON ADLS   │
│   DROP SCHEMA silver            →  schema deleted, files still on ADLS     │
│                                                                             │
│   Use when: ADF, Synapse, or other tools also read the same ADLS files     │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│                    VOLUME — same split applies                              │
│                                                                             │
│   Managed Volume  → UC owns storage → drop = files deleted                 │
│   External Volume → you own ADLS    → drop = files PRESERVED               │
│                                                                             │
│   bronze-volume in our project = EXTERNAL Volume                           │
│   So DELETE FILE via UI = deletes the real ADLS file immediately           │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## When Will Deletion NOT Touch Your ADLS Files?

| Action | Managed Table/Volume | External Table/Volume |
|---|---|---|
| `DROP TABLE` | Deletes metadata + ADLS files | Deletes metadata only — files safe |
| `DROP SCHEMA` | Deletes schema + all tables + all files | Deletes schema + table metadata — files safe |
| `DROP VOLUME` | Deletes volume + all files inside | Deletes volume registration — files safe |
| Delete file via Volume UI | Deletes real ADLS file (same for both) | Same — Volume UI always touches real storage |
| `TRUNCATE TABLE` | Deletes all rows (files deleted for managed) | Deletes all rows (files deleted for external too) |

**Key insight:** The DROP behaviour differs between managed and external.  
But deleting a file directly via the Volume UI **always** touches the real ADLS file — regardless of managed or external.

---

## Our Project — Which Is Which?

| Object | Type | DROP behaviour |
|---|---|---|
| `bronze-volume` | External Volume | Files stay on ADLS |
| `pipeline_audit` (if created as managed) | Managed Table | Files deleted on DROP |
| Silver `charging_sessions` (planned) | External Table | Files stay on ADLS |


## Setup — Set catalog and schema first

Always run this before any SQL in this notebook.

In [ ]:
spark.sql("USE CATALOG dbw_ev_intelligence_dev")
spark.sql("USE SCHEMA default")

print("Catalog :", spark.sql("SELECT current_catalog()").collect()[0][0])
print("Schema  :", spark.sql("SELECT current_schema()").collect()[0][0])

---

# PART A — MANAGED TABLE

## Cell A1 — Create a Managed Table and insert data

Unity Catalog picks the storage location automatically (its own managed storage root).  
You do NOT specify a `LOCATION` — that is what makes it managed.

In [ ]:
# Create managed table — no LOCATION specified, UC decides where files go
spark.sql("""
    CREATE TABLE IF NOT EXISTS ev_stations_managed (
        station_id   STRING,
        station_name STRING,
        city         STRING,
        capacity_kw  INT
    )
    USING DELTA
""")

# Insert sample rows
spark.sql("""
    INSERT INTO ev_stations_managed VALUES
        ('ST001', 'MG Road Charger',   'Bengaluru', 150),
        ('ST002', 'Andheri Fast Charge','Mumbai',    200),
        ('ST003', 'Cyber Hub Station', 'Gurugram',  120)
""")

print("Managed table created and data inserted.")
spark.sql("SELECT * FROM ev_stations_managed").show()

## Cell A2 — Find where the managed table files actually live

UC stores them in its own managed storage — you did not choose this path.

In [ ]:
# DESCRIBE EXTENDED shows the full table detail including storage location
detail = spark.sql("DESCRIBE DETAIL ev_stations_managed").collect()[0]

print("Table type :", detail['tableType'])   # should print: MANAGED
print("Location   :", detail['location'])    # UC-managed path — NOT your ADLS container
print("Format     :", detail['format'])      # delta
print()
print("Notice: the location is a UC-managed path, not abfss://bronze@evdatalakedev...")
print("UC owns these files. When you DROP this table, these files will be deleted.")

## Cell A3 — DROP the managed table and observe what happens

After this cell, query the table — it will not exist.  
The Delta files at the managed location are also gone.

In [ ]:
# Get location BEFORE dropping so we can try to read after
location_before_drop = spark.sql("DESCRIBE DETAIL ev_stations_managed").collect()[0]['location']
print("Files were at:", location_before_drop)
print()

# DROP the managed table
spark.sql("DROP TABLE ev_stations_managed")
print("Table dropped.")
print()

# Try to read the files directly — they should be GONE
try:
    files = dbutils.fs.ls(location_before_drop)
    print(f"Files still exist: {len(files)} item(s) found")
    print("Unexpected — managed table files should have been deleted.")
except Exception:
    print("Files NOT found at the managed location — confirmed deleted by DROP.")
    print()
    print("LESSON: DROP TABLE on a MANAGED table = metadata gone + ADLS files gone.")

---

# PART B — EXTERNAL TABLE

## Cell B1 — Write sample data to a specific ADLS path first

For an external table, the files must exist at a path YOU control.  
We write them to the Bronze Volume first — this becomes the external table's location.

In [ ]:
from pyspark.sql import Row

EXTERNAL_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/demo/ev_stations_external/"

# Create sample data
rows = [
    Row(station_id="ST004", station_name="Koramangala Hub",  city="Bengaluru", capacity_kw=180),
    Row(station_id="ST005", station_name="Bandra Supercharge",city="Mumbai",    capacity_kw=250),
    Row(station_id="ST006", station_name="DLF Cyber City",   city="Gurugram",  capacity_kw=100),
]
df = spark.createDataFrame(rows)

# Write as Delta to a path WE control on the Bronze Volume
df.write.format("delta").mode("overwrite").save(EXTERNAL_PATH)

print(f"Data written to: {EXTERNAL_PATH}")
print()

# Verify files exist
files = dbutils.fs.ls(EXTERNAL_PATH)
print(f"{len(files)} item(s) at the path (Delta log + parquet files):")
for f in files:
    print(f"  {f.path}")

## Cell B2 — Create an External Table pointing to that path

The key difference from managed: we specify `LOCATION`.  
UC registers the metadata but does NOT touch or own the files.

In [ ]:
# For external tables, LOCATION must be an abfss:// path (not /Volumes/...)
# /Volumes/... is a notebook convenience path — LOCATION needs the real ADLS URI
EXTERNAL_ADLS_PATH = "abfss://bronze@evdatalakedev.dfs.core.windows.net/demo/ev_stations_external/"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS ev_stations_external
    USING DELTA
    LOCATION '{EXTERNAL_ADLS_PATH}'
""")

print("External table created — pointing to your ADLS path.")
print()
spark.sql("SELECT * FROM ev_stations_external").show()

## Cell B3 — Confirm it is EXTERNAL and where the files are

In [ ]:
detail = spark.sql("DESCRIBE DETAIL ev_stations_external").collect()[0]

print("Table type :", detail['tableType'])   # should print: EXTERNAL
print("Location   :", detail['location'])    # your abfss://bronze@evdatalakedev... path
print("Format     :", detail['format'])      # delta
print()
print("Notice: location = YOUR ADLS path. UC does not own these files.")
print("Dropping this table will remove UC's metadata but leave files on ADLS.")

## Cell B4 — DROP the external table and prove files survive

This is the core proof: DROP TABLE on an external table does NOT delete your ADLS files.

In [ ]:
files_before = dbutils.fs.ls(EXTERNAL_PATH)
print(f"Files BEFORE drop: {len(files_before)} item(s) at {EXTERNAL_PATH}")
print()

# DROP the external table
spark.sql("DROP TABLE ev_stations_external")
print("Table dropped from Unity Catalog.")
print()

# Try to query it — should fail (metadata is gone)
try:
    spark.sql("SELECT * FROM ev_stations_external").show()
except Exception as e:
    print("Query failed (expected) — table no longer exists in UC metadata.")
    print()

# Check files on ADLS — should still be there
try:
    files_after = dbutils.fs.ls(EXTERNAL_PATH)
    print(f"Files AFTER drop: {len(files_after)} item(s) still at {EXTERNAL_PATH}")
    print()
    print("LESSON: DROP TABLE on an EXTERNAL table = metadata gone, FILES PRESERVED.")
    print("You can re-register the table anytime with the same CREATE TABLE ... LOCATION command.")
except Exception:
    print("Files not found — unexpected for an external table drop.")

## Cell B5 — Re-register the external table from the same files

Because files are preserved, you can bring the table back instantly — no data loss, no reload.

In [ ]:
spark.sql(f"""
    CREATE TABLE ev_stations_external
    USING DELTA
    LOCATION '{EXTERNAL_ADLS_PATH}'
""")

print("Table re-registered from the same ADLS path — data is back instantly.")
print()
spark.sql("SELECT * FROM ev_stations_external").show()

## Cell B6 — Cleanup: drop the demo external table and files

In [ ]:
spark.sql("DROP TABLE IF EXISTS ev_stations_external")
dbutils.fs.rm(EXTERNAL_PATH, recurse=True)
print("Demo table and demo files cleaned up.")

---

# PART C — VOLUME BEHAVIOUR (Why Your File Was Deleted)

## Cell C1 — Check whether bronze-volume is Managed or External

In [ ]:
volume_info = spark.sql("""
    DESCRIBE VOLUME dbw_ev_intelligence_dev.default.`bronze-volume`
""").collect()

for row in volume_info:
    print(f"{row[0]:20s} : {row[1]}")

print()
print("If volume_type = EXTERNAL → files on ADLS, DELETE via UI = real ADLS file deleted")
print("If volume_type = MANAGED  → files in UC storage, DROP VOLUME = files deleted")

## Cell C2 — Why deleting via Volume UI always touches real ADLS

Explanation of what happened when you deleted the file.

In [ ]:
print("=" * 65)
print("WHY YOUR FILE WAS DELETED FROM ADLS")
print("=" * 65)
print()
print("You deleted a file via the Catalog UI:")
print("  Catalog → bronze-volume → realtime → ... → file → delete")
print()
print("What actually happened:")
print("  The Volume UI is a file browser for your ADLS path.")
print("  Deleting a file there = calling dbutils.fs.rm() on the real ADLS file.")
print("  There is no recycle bin. The file is immediately gone from ADLS.")
print()
print("This happens regardless of Managed vs External Volume.")
print("The Managed/External distinction only affects DROP VOLUME behaviour, not file deletion.")
print()
print("=" * 65)
print("WHEN WILL DELETION NOT TOUCH YOUR ADLS FILES?")
print("=" * 65)
print()
print("Only one scenario: DROP TABLE or DROP VOLUME on an EXTERNAL object.")
print()
print("  DROP TABLE ev_stations_external  → UC metadata removed, ADLS files safe")
print("  DROP VOLUME bronze-volume        → Volume deregistered, ADLS files safe")
print()
print("Everything else (delete file via UI, dbutils.fs.rm, TRUNCATE TABLE)")
print("touches the actual ADLS storage immediately.")

---

# PART D — COMPLETE COMPARISON SUMMARY

## Cell D1 — Final reference card: every action and its outcome

In [ ]:
print("=" * 70)
print("MANAGED vs EXTERNAL — WHAT HAPPENS TO YOUR DATA")
print("=" * 70)
print()
print(f"{'Action':<40} {'Managed':<15} {'External'}")
print("-" * 70)
rows = [
    ("DROP TABLE",                      "meta + FILES gone", "meta gone, FILES SAFE"),
    ("DROP SCHEMA (CASCADE)",           "meta + FILES gone", "meta gone, FILES SAFE"),
    ("DROP VOLUME",                     "meta + FILES gone", "meta gone, FILES SAFE"),
    ("TRUNCATE TABLE",                  "all rows deleted",  "all rows deleted"),
    ("DELETE FROM table WHERE ...",     "rows deleted",      "rows deleted"),
    ("Delete file via Volume UI",       "file deleted",      "file deleted"),
    ("dbutils.fs.rm(path)",             "file deleted",      "file deleted"),
    ("Re-register after DROP TABLE",    "not possible",      "yes — same LOCATION"),
]
for action, managed, external in rows:
    print(f"  {action:<38} {managed:<15} {external}")

print()
print("=" * 70)
print("WHEN TO USE WHICH")
print("=" * 70)
print()
print("  MANAGED TABLE  — use when:")
print("    - Only Databricks reads/writes this data")
print("    - You want UC to fully manage lifecycle (create → drop = clean)")
print("    - Temporary/intermediate tables during transformations")
print()
print("  EXTERNAL TABLE — use when:")
print("    - Files already exist on ADLS (e.g. ADF wrote them)")
print("    - ADF, Synapse, or Power BI also read the same files")
print("    - You want DROP TABLE to be a safe metadata-only operation")
print("    - Bronze/Silver/Gold Delta tables in our project (ADF writes Bronze)")
print()
print("  EXTERNAL VOLUME — use when:")
print("    - You want a notebook-friendly path (/Volumes/...) to your ADLS data")
print("    - ADF or other tools also write to the same ADLS container")
print("    - Our bronze-volume: ADF writes pipeline_audit.csv, notebook reads it")
print()
print("=" * 70)
print("OUR PROJECT MAPPING")
print("=" * 70)
print()
print("  bronze-volume           EXTERNAL Volume  — files on your ADLS bronze/")
print("  Silver charging_sessions EXTERNAL Table  — Delta files on your ADLS silver/")
print("  Gold aggregations       EXTERNAL Table   — Delta files on your ADLS gold/")
print("  Temp/staging tables     MANAGED Table    — UC manages, safe to DROP freely")